# CapOpt — OOPSLA 2026 Artifact Evaluation

Welcome to the CapOpt artifact. This notebook is the primary review interface, with all evaluation commands available as executable cells—no separate terminal required.

The core `capopt` package is the prebuilt library in `artifact/lib` (CPython 3.12 bytecode plus a native extension), while all orchestration, evaluation, benchmark, and test code is readable in `artifact/source`.

## 0. Notebook helper

The next cell locates the artifact root and defines `run(...)`, which streams a command's output into the notebook without shell interpolation.

You can also use `run(...)` for your own commands — for example `run(sys.executable, "ae/runner.py", "claims", "--strict")`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "ae" / "runner.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "ae" / "runner.py").is_file(), "Run this notebook from the AE package"

IMPORT_PATHS = [
    ROOT / "artifact" / "lib",
    ROOT / "artifact" / "source",
    ROOT / "artifact" / "source" / "superoptimization",
]
for candidate in reversed(IMPORT_PATHS):
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(path) for path in IMPORT_PATHS]
    + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else [])
)

def run(*args, check=True, cwd=None):
    command = [str(arg) for arg in args]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd or ROOT, check=check)

print(f"Artifact root: {ROOT}")

### The prebuilt library and provenance

The next cell imports the optimizer and reports where it came from. Expected: `capopt` resolves inside `artifact/lib` (the prebuilt capopt library: CPython 3.12 bytecode built in the main repository).

In [ ]:
import capopt
from capopt import _native

source_record = json.loads((ROOT / "artifact" / "SOURCE.json").read_text())
core = source_record["core_library"]
print(f"capopt {capopt.__version__} from {capopt.__file__}")
print(f"native backend: {_native.backend_name()} (HAVE_NATIVE={_native.HAVE_NATIVE})")
print(f"built with CPython {core['python_version_used_to_compile']} "
      f"({core['bytecode_tag']}) from commit {source_record['git_commit'][:12]}")
assert Path(capopt.__file__).is_relative_to(ROOT / "artifact" / "lib"), \
    "capopt must resolve to the prebuilt library in artifact/lib"

## 1. Setup and integrity (seconds)

Run the next cell to validate the packaged environment and the artifact's closure: the dedicated Python venv, the required packages, Z3, the CapOpt import, the vendored provenance record, symlink containment, and the licensing boundary.

**Expected output:** the checks end with `ENVIRONMENT: PASS` and `CLOSURE: PASS`. **Estimated time:** seconds.

In [ ]:
run(sys.executable, "ae/runner.py", "env")
run(sys.executable, "ae/runner.py", "closure")

### 1.1 Morello hardware spot-check (about 1 minute)

The hardware-dependent testing later in this notebook needs Morello board access, so it is checked here, up front. 

**Expected output:** `MORELLO P01: PASS` with a counter report — or a documented
`SKIPPED` line when no board is reachable.

In [ ]:
MORELLO_BOARD = "morello-1"  # use "morello-2" (192.168.10.102) if needed


def _morello_report(run_dir_rel):
    run(sys.executable, "ae/runner.py", "morello-report", "--run-dir", run_dir_rel)
    report = json.loads((ROOT / run_dir_rel / "report.json").read_text())
    display(report)


status_result = subprocess.run(
    [sys.executable, "ae/morello/request.py", "status"],
    cwd=ROOT, text=True, capture_output=True, check=False,
)
if status_result.returncode == 0:
    # Route 1 — containerized: bounded request through the host bridge.
    request_command = [
        sys.executable, "ae/morello/request.py", "measure",
        "--board", MORELLO_BOARD, "--iterations", "10", "--warmup", "1",
    ]
    print("$", " ".join(request_command), flush=True)
    request_result = subprocess.run(
        request_command, cwd=ROOT, text=True, capture_output=True, check=False
    )
    print(request_result.stdout)
    if request_result.stderr:
        print(request_result.stderr, file=sys.stderr)
    if request_result.returncode:
        print("MORELLO P01: SKIPPED (optional board access unavailable)")
    else:
        morello_response = json.loads(request_result.stdout)
        print(morello_response.get("output", ""))
        _morello_report(morello_response["run_dir"])
else:
    # Route 2 — local run (no bridge): call the host-only collector directly;
    # it uses this machine's OpenSSH client and the pinned known_hosts.
    from datetime import datetime

    run_dir_rel = (
        "ae-output/morello/spotcheck-" + datetime.now().strftime("%Y%m%d-%H%M%S")
    )
    collect_command = [
        sys.executable, "ae/morello/collect.py", "--board", MORELLO_BOARD,
        "--iterations", "10", "--warmup", "1", "--run-dir", run_dir_rel,
    ]
    print("$", " ".join(collect_command), flush=True)
    collect_result = subprocess.run(
        collect_command, cwd=ROOT, text=True, capture_output=True, check=False
    )
    print(collect_result.stdout)
    if collect_result.stderr:
        print(collect_result.stderr, file=sys.stderr)
    if collect_result.returncode:
        print("MORELLO P01: SKIPPED (no host bridge and no direct board access)")
    else:
        _morello_report(run_dir_rel)


## 2. Kick the tires (<10 minutes)

Run the next cell to exercise the artifact end to end, such as fast verifier/optimizer etc.

**Expected output:** the final line is `KICK-THE-TIRES: PASS`.

In [ ]:
run(sys.executable, "ae/runner.py", "kick-the-tires")

## 3. Paper-result testings on small-size datasets (about 5 minutes)

This section tests the paper's four headline experiments:

| Part | Paper | What is tested |
|---|---|---|
| 3.1 | Section 6.1 | Performance improvements (Hacker's Delight, SPEC CPU2017, LLaMA.cpp) |
| 3.2 | Section 6.2 | Two-level vs. single-level optimization (ablation study, Table 2) |
| 3.3 | Section 6.4 | Comparison with Souper |
| 3.4 | Section 6.4 | Superoptimization starting from `-O3` (Table 4) |

Every part runs the repository's **real analysis pipeline** — the same
aggregation, gating, and summarization scripts that produced the paper's
numbers — on small-size input. The full
campaigns take hours to days, so the
numbers here are not the paper's full-size numbers; what each part must
reproduce is the paper's **qualitative claim**, which has to hold on the small
data too.


### 3.1 Paper Section 6.1 — Performance improvements

Section 6.1 claims CapOpt delivers consistent speedups over CHERI-LLVM `-O3`
across all three benchmark families — Hacker's Delight, SPEC CPU2017, and
LLaMA.cpp, on both ARM Morello and CHERI-RISC-V across all deployed rewrites (every rewrite is SMT-proven and
empirically validated before deployment).

Run the next cell to execute the Section 6.1 aggregation pipeline on the
bundled small campaign, in three lanes: the Hacker's Delight speedup rows
(C2), the SPEC CPU2017 and LLaMA.cpp speedup rows (C3), and the empirical
no-regression validation gate (C9). On the small dataset the speedup
magnitudes differ from the paper's full-campaign tables, but every benchmark
family must show a positive improvement.

**Expected output:** a `PASS` for each of the three lanes. **Estimated time:**
about 2 minutes.


In [ ]:
run(sys.executable, "ae/runner.py", "reproduce-small", "--claim", "C2")
run(sys.executable, "ae/runner.py", "reproduce-small", "--claim", "C3")
run(sys.executable, "ae/runner.py", "reproduce-small", "--claim", "C9")


### 3.2 Paper Section 6.2 — Two-level vs. single-level optimization

Section 6.2 (Table 2) claims the combined two-level pipeline (IR-level then
assembly-level synthesis) outperforms IR-only and assembly-only synthesis in
every benchmark category on both platforms — the paper's evidence that the two
levels find complementary optimizations.

**Expected output:** a `PASS` for the C1 lane. **Estimated time:** under a
minute.


In [ ]:
run(sys.executable, "ae/runner.py", "reproduce-small", "--claim", "C1")

### 3.3 Paper Section 6.4 — Comparison with Souper

The paper compares CapOpt against Souper on conventional non-CHERI RISC-V, where both tools apply, and claims comparable speedups differing by at most ±2% per kernel. 

**Expected output:** `C6 RECOMPUTE: PASS`, with a per-kernel and geomean CapOpt-vs-Souper gap across all 25 kernels — with symmetric lanes and  effective rewrites. **Estimated time:** seconds.

In [ ]:
run(sys.executable, "ae/runner.py", "recompute", "--claim", "C6")
c6 = json.loads((ROOT / "ae-output/recompute/c6.json").read_text())
{
    "fresh_geomean": c6["fresh"]["geomean"],
    "fresh_verdict": c6["fresh"]["verdict"],
    "fields_excluded_from_comparison": c6["fields_excluded_from_comparison"],
    "section_mismatches": c6["section_mismatches"],
    "passed": c6["passed"],
}

### 3.4 Paper Section 6.4 — Superoptimization starting from `-O3` (Table 4)

**Expected output:** `C7 RECOMPUTE: PASS`. **Estimated time:** seconds.

In [ ]:
run(sys.executable, "ae/runner.py", "recompute", "--claim", "C7")
c7 = json.loads((ROOT / "ae-output/recompute/c7.json").read_text())
{
    "fresh": c7["fresh"],
    "per_kernel_mismatches": c7["per_kernel_mismatches"],
    "median_check_failures": c7["median_check_failures"],
    "recorded_evidence_audit": {
        key: c7["recorded_evidence_audit"][key]
        for key in (
            "kernels_with_applied_rewrites",
            "kernels_text_differing",
            "suite_smt_proven",
        )
    },
    "passed": c7["passed"],
}

## 4. Reuse: try CapOpt on a new input (about one minute)

CapOpt is a tool contribution, so this cell demonstrates it on input that is **not** part of any packaged experiment. Run the next cell to write a brand-new CHERI IR function to `ae-output/reuse/new_input.ll` and launch the packaged `capopt` command line on it. The optimizer's complete candidate/verifier log (about a hundred thousand lines of solver activity) is preserved at `ae-output/reuse/capopt.log`; the cell displays the CLI summary and the optimized IR.

**Expected output:** `✓ Optimization completed`, and the optimized IR replaces the three-instruction permission chain (`perms.get` → `and` → `perms.set`) with a single `llvm.cheri.cap.perms.and` intrinsic — the same permission-restriction fusion whose Z3 equivalence proof is exercised in Section 2.

**To go further:** edit `new_ir` to any LLVM IR function you like (or point `--input` at your own `.ll` file) and rerun the cell; `--arch aarch64-morello` selects the Morello backend, and `capopt --help` lists the search-budget knobs. The verifier fails closed: candidates that cannot be proven equivalent are not applied.

In [ ]:
new_ir = """define i8 addrspace(200)* @reviewer_demo(i8 addrspace(200)* %cap, i64 %mask) {
entry:
  %perms = call i64 @llvm.cheri.cap.perms.get(i8 addrspace(200)* %cap)
  %masked = and i64 %perms, %mask
  %tight = call i8 addrspace(200)* @llvm.cheri.cap.perms.set(i8 addrspace(200)* %cap, i64 %masked)
  ret i8 addrspace(200)* %tight
}"""
reuse_dir = ROOT / "ae-output" / "reuse"
reuse_dir.mkdir(parents=True, exist_ok=True)
(reuse_dir / "new_input.ll").write_text(new_ir + "\n")
command = [
    "capopt", "--quiet",
    "--input", "ae-output/reuse/new_input.ll",
    "--function", "reviewer_demo",
    "--arch", "riscv64-cheri",
    "--output", "ae-output/reuse/optimized.ll",
]
print("$", " ".join(command), flush=True)
result = subprocess.run(
    command,
    cwd=ROOT,
    text=True,
    capture_output=True,
    # Keep the CLI's variant-store bundles out of the checksummed tree,
    # exactly as every ae/runner.py lane does.
    env={**os.environ, "CAPOPT_ARTIFACT_ROOT": str(reuse_dir / "capopt-artifacts")},
)
log_lines = (result.stdout + result.stderr).splitlines()
(reuse_dir / "capopt.log").write_text("\n".join(log_lines) + "\n")
if result.returncode:
    print("\n".join(log_lines[-30:]))
    raise RuntimeError(f"capopt exited with {result.returncode}; see ae-output/reuse/capopt.log")
summary, warnings = [], {}
for line in log_lines:
    if " - INFO - " in line or " - DEBUG - " in line:
        continue
    if " - WARNING - " in line:
        message = line.split(" - WARNING - ", 1)[1]
        warnings[message] = warnings.get(message, 0) + 1
        continue
    summary.append(line)
print("\n".join(summary))
if warnings:
    print("\nOptimizer warnings (deduplicated; both are expected in this container):")
    for message, count in warnings.items():
        print(f"  {count}x {message}")
print(f"\nComplete search/verifier log ({len(log_lines):,} lines): ae-output/reuse/capopt.log")
print("\nOptimized IR:")
print((reuse_dir / "optimized.ll").read_text())

## 5. Full performance reproduction (hours to days)

| Case | Launch script |
|---|---|
| 5.1 Hacker's Delight on CHERI-RISC-V/QEMU | `scripts/run_all.sh --arch riscv-cheri --benches hacker` |
| 5.2 Morello campaign | `scripts/run_campaign.sh --arch arm-morello` |
| 5.3 SPEC CPU2017 | `scripts/run_all.sh --arch riscv-cheri --benches spec` |


In [ ]:
long_route_prerequisites = {
    "CHERI_LLVM_BIN (CHERI-LLVM clang/lld bin directory)": os.environ.get("CHERI_LLVM_BIN"),
    "CHERI_SYSROOT (CheriBSD purecap sysroot)": os.environ.get("CHERI_SYSROOT"),
    "qemu-system-riscv64cheri on PATH (full-system QEMU-CHERI)": shutil.which("qemu-system-riscv64cheri"),
    "MORELLO_LLVM_BIN (Morello-LLVM bin directory)": os.environ.get("MORELLO_LLVM_BIN"),
    "MORELLO_HOSTS (ssh-reachable Morello boards)": os.environ.get("MORELLO_HOSTS"),
    "licensed SPEC tree at artifact/source/benchmarks/spec-cpu2017-1.1.0": (
        str(ROOT / "artifact/source/benchmarks/spec-cpu2017-1.1.0")
        if (ROOT / "artifact/source/benchmarks/spec-cpu2017-1.1.0").is_dir()
        else None
    ),
}
for name, value in long_route_prerequisites.items():
    print(f"  {'present' if value else 'absent':7}  {name}")
print(
    "\nIn the closed image all six are absent by design; in the full image"
    "\n(capopt-oopsla26-ae:full) the CHERI-LLVM, sysroot, and QEMU entries are"
    "\npresent, and running bin/setup-qemu-image.sh once prepares the guest"
    "\ndisk. The launch cells below print their commands and only execute"
    "\nwhere their prerequisites are present."
)

### 5.1 Hacker's Delight on CHERI-RISC-V/QEMU (about 1 hour smoke)

This is the cheapest full re-measurement: the 25 public Hacker's Delight patterns build with the pinned CHERI-LLVM for each pipeline (`baseline`, `ir-only`, `asm-only`, `capopt`) and are measured in **one full-system QEMU-CHERI boot** via the guest retired-instruction dispatcher.

**Expected output on Run All:** the printed launch command only (if no SDK). **Estimated time when launched:** about 1 hour with the shown smoke settings (`--iterations 3 --warmup 1`); several hours for the full 10-iteration sweep. 

In [ ]:
RUN_QEMU_HD_SWEEP = False  # about 1 hour with these smoke settings; several hours in full
sweep_command = [
    "bash", "scripts/run_all.sh", "--arch", "riscv-cheri", "--benches", "hacker",
    "--iterations", "3", "--warmup", "1",  # smoke settings; drop both for the full 10-iteration sweep
]
if not RUN_QEMU_HD_SWEEP:
    print("Not launched in this session. On a prepared host, set RUN_QEMU_HD_SWEEP = True to run:")
    print("$ cd artifact/source &&", " ".join(sweep_command))
elif not (os.environ.get("CHERI_LLVM_BIN") and os.environ.get("CHERI_SYSROOT")):
    print("Refusing to launch: CHERI_LLVM_BIN and CHERI_SYSROOT are not set;")
    print("see the preflight cell above and REPRODUCING.md section 1.")
else:
    guest_img = os.environ.get("QEMU_CHERI_IMAGE", "")
    if guest_img and not Path(guest_img).exists():
        # Full image: decompress the packaged CheriBSD guest disk once.
        run("bash", ROOT / "bin" / "setup-qemu-image.sh")
    run(*sweep_command, cwd=ROOT / "artifact/source")

### 5.2 Morello hardware campaign (hours to days)

The dated-campaign script builds with Morello-LLVM, stages the binaries to the boards in `MORELLO_HOSTS` (round-robined), measures each benchmark ten times pinned to a non-housekeeping core under `pmcstat`.

**Expected output on Run All:** the printed launch commands only. **Estimated time when launched:** hours for the Hacker's Delight suite, days with SPEC/LLaMA included.

In [ ]:
RUN_MORELLO_CAMPAIGN = False  # hours to days; requires AEC-arranged board access
campaign_command = ["bash", "scripts/run_campaign.sh", "--arch", "arm-morello"]
staging_dry_run = [
    "bash", "scripts/morello_remote_run.sh",
    "--benches", "hacker", "--pipelines", "capopt", "--dry-run",
]
if not RUN_MORELLO_CAMPAIGN:
    print("Not launched in this session. On a prepared host, set RUN_MORELLO_CAMPAIGN = True to run:")
    print("$ cd artifact/source &&", " ".join(campaign_command))
    print("Staging sanity check without ssh:")
    print("$ cd artifact/source &&", " ".join(staging_dry_run))
elif not os.environ.get("MORELLO_HOSTS"):
    print("Refusing to launch: set MORELLO_HOSTS to your AEC-granted account list")
    print('first, for example: export MORELLO_HOSTS="aec@192.168.10.101".')
else:
    run(*campaign_command, cwd=ROOT / "artifact/source")

### 5.3 SPEC CPU2017 (hours to days; licensed installation required)

SPEC sources are never redistributed. To re-measure the SPEC cells, place — or symlink — your own licensed SPEC CPU2017 1.1.0 tree at `artifact/source/benchmarks/spec-cpu2017-1.1.0` (the build scripts read the workloads from that fixed path), then flip the flag in the next cell.

**Expected output on Run All:** the printed launch command only. 

In [ ]:
RUN_SPEC_SWEEP = False  # licensed SPEC CPU2017 1.1.0 required
spec_command = ["bash", "scripts/run_all.sh", "--arch", "riscv-cheri", "--benches", "spec"]
spec_tree = ROOT / "artifact/source/benchmarks/spec-cpu2017-1.1.0"
if not RUN_SPEC_SWEEP:
    print("Not launched in this session. On a prepared host, set RUN_SPEC_SWEEP = True to run:")
    print("$ cd artifact/source &&", " ".join(spec_command))
elif not spec_tree.is_dir():
    print("Refusing to launch: no licensed SPEC tree at", spec_tree)
    print("Place (or symlink) your own SPEC CPU2017 1.1.0 installation there first;")
    print("it is never packaged, redistributed, or archived by this artifact.")
else:
    run(*spec_command, cwd=ROOT / "artifact/source")